In [1]:
from scipy.sparse import load_npz
from pathlib import Path

import pandas as pd
import numpy as np
import re

from sklearn.cluster import DBSCAN

from sklearn.metrics import silhouette_score
from sklearn.model_selection import ParameterGrid

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

import joblib
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
processed_dir = Path("../data/processed")
results_dir = Path("../results/clustering")
model_dir = Path("../models")

matrice_tfidf_bigrammes = load_npz(processed_dir / "tfidf_matrix_bigrammes.npz")
model = joblib.load(model_dir / "dbscan_champion.pkl")
labels = pd.read_csv(results_dir / "champion_labels.csv")
ids = pd.read_csv(processed_dir / "tfidf_row_ids.csv")
data = pd.read_csv(processed_dir / "sample_clean.csv")


assert matrice_tfidf_bigrammes.shape[0] == len(ids)
assert ids["id"].is_unique
assert ids["id"].equals(data["id"])
assert labels["id"].equals(ids["id"])
assert np.array_equal(labels["cluster"].to_numpy(), model.labels_)

In [3]:
clusters = ids.assign(
    cluster=model.labels_,
    products=data["description_texte"],
)

In [61]:
pd.set_option("display.max_colwidth", None)
clusters

,id,cluster,products
0,1,0,"active classic boxers - s reason boxers cult favorite - cool , especially sticky situations . quick - drying , lightweight underwear takes minimal space travel pack . exposed , brushed waistband offers - - skin softness , - panel construction traditional boxer classic fit , functional fly . 3.7 - oz 100 % recycled polyester moisture - wicking performance . inseam size m 4 1/2 . recyclable common threads recycling program ."
1,2,-1,"active sport boxer briefs - skinning glory requires movement boxers deciding poach route . form - fitting active sport boxer briefs breathable 93 % polyester 71 % recycled fabric s fast - wicking , dries quickly 7 % spandex stretch seamless waistband soft leg edges won t roll bind . gusseted , flat - sewn 6 inseam size m offset prevent inner - thigh chafe . fly - free smooth panel . recyclable common threads recycling program ."
2,3,-1,"active sport briefs - superbreathable - fly briefs minimalist s choice high - octane endeavors . blend fast - wicking , quick - drying 93 % polyester 71 % recycled 7 % spandex stretch - mesh support open mesh cooling airflow . soft edging leg openings seamless waist won t roll create friction layers . smooth panel opacity . recyclable common threads recycling program ."
3,4,-1,"alpine guide pants - skin , climb ice , switch rock , traverse knife - edge ridge boogie - durable , weather - resistant breathable soft - shell pants stride mountain endeavor . midweight stretch - woven polyester won t restrict moves , brushed interior maintains - - skin comfort . 2 - way zippered fly keeps things easy harness , gusseted , zippered cuffs 2 - position snaps tie - loops . water - resistant zippered pockets , thigh , hip . recyclable common threads recycling program ."
4,5,14,"alpine wind jkt - high ridges , steep ice alpine , jacket serves true best worlds staple . excels stand - shell blustery rock climbs , cool - weather trail runs high - output ski tours . , conditions ice alpine climbing , functions lightly insulated windshirt approach , frictionless midlayer s time bundle tie - . polyester ripstop shell deluge dwr durable water repellent treatment sheds snow blocks wind , smooth , lightly brushed hanging mesh liner wicks moisture , dries fast , doesn t bind baselayers . superlight stretch - woven underarm panels enhance breathability allow unimpaired arm motion , hand pockets close zippers . drawcord hem , elastic cuffs , heat - transfer reflective logo regular - coil , center - zipper dwr finish round features . updated season improved fit . recyclable common threads recycling program ."
...,...,...,...
495,496,0,"cap 2 bottoms - cut loose maddening crowds search undone . capilene 2 fabric balances fast - drying performance lightweight insulation haven t. open - knit 4 - oz polyester 54 % recycled provides exceptional breathability , wicks perspiration , dries flash , pants ultimate comfort beneath layers chilly days . feature elastic waistband s brushed softness , seams set forward eliminate chafe . gusseted crotch , functional fly gladiodor natural odor control garment . recyclable common threads recycling program ."
496,497,0,"cap 2 crew - crew takes edge fickle weather . clearing conditions , fastest dry time performance baselayer fabrics , open - knit construction breathes wicks dry cool . weather window shuts , wear insulation beneath layers . self - fabric neck s soft skin , raglan sleeves flat - ribbed underarm panels merge way shoulder straps eliminate chafe . 4 - oz 100 % polyester 54 % recycled gladiodor natural odor control garment . recyclable common threads recycling program ."
497,498,1,"- time shell - need use morning times umbrella . - times handsome matte - finish , 100 % recycled polyester shell provides tasteful sports - coat - length coverage vicissitudes weather city country . hardy 2.5 - layer , 4.2 - oz polyester ripstop h2no waterproof / breathable barrier deluge dwr durable water repellent finish scoffs cloudbursts 100

In [ ]:
def find_cluster_items(item_id, product_name):
    """
    Cette fonction prend un identifiant de produit en entrée et renvoie les identifiants des produits
    appartenant au même cluster que le produit donné.
    """
    # Vérifier si l'identifiant du produit est présent dans le DataFrame clusters
    if item_id not in clusters["id"].values:
        raise ValueError(f"L'identifiant du produit {item_id} n'existe pas dans le DataFrame clusters.")
    
    # Récupérer le cluster de l'id donné
    item_cluster = clusters.loc[clusters["id"] == item_id, "cluster"].values[0]
    item_cluster_name = clusters.loc[clusters["id"] == item_id, "products"].values[0]
    
    # Récupérer les identifiants des produits appartenant au même cluster
    cluster_items = clusters.loc[clusters["cluster"] == item_cluster, "id"].tolist()
    
    return cluster_items

In [12]:
find_cluster_items(int(input("Entrez un identifiant produit : ")))

[8, 219, 220, 221, 240, 254, 255, 261, 262, 291]

In [68]:
def item_id(saisie):
    """
    Cette fonction prend une saisie utilisateur en entrée et renvoie l'identifiant du produit correspondant.
    """
    # Vérifier si la saisie est un entier
    if not saisie.isdigit():
        raise ValueError("La saisie doit être un entier.")
    
    item_id = int(saisie)
    
    # Vérifier si l'identifiant du produit est présent dans le DataFrame clusters
    if item_id not in clusters["id"].values:
        raise ValueError(f"L'identifiant du produit {item_id} n'existe pas dans le DataFrame clusters.")
    
    return item_id

In [71]:
def item_name_to_id(saisie):
    """
    Cette fonction prend un nom de produit en entrée et renvoie l'identifiant du produit correspondant.
    """
    # Vérifier si le nom du produit est présent dans le DataFrame clusters
    if saisie not in clusters["products"].values:
        raise ValueError(f"Le nom du produit '{saisie}' n'existe pas dans le DataFrame clusters.")
    
    item_id = clusters.loc[clusters["products"] == saisie, "id"].values[0]
    
    return item_id

In [96]:
def recommander(saisie):

    # Récupérer le cluster de l'id donné
    if saisie.isdigit():
        item_id = int(saisie)
    else:
        item_id = item_name_to_id(saisie)

    input_item = clusters[clusters["id"] == item_id]

    # Récupérer les produits de ce cluster
    cluster_items = clusters.loc[clusters["cluster"] == input_item["cluster"].values[0]].copy()

    cluster_items["score"] = cosine_similarity(
            matrice_tfidf_bigrammes[[input_item.index[0]]],
            matrice_tfidf_bigrammes[cluster_items.index.to_numpy()],
        ).ravel()    

    cluster_items.sort_values(by="score", ascending=False, inplace=True)
    recommandation = cluster_items.drop(index=input_item.index).head(5)

    return recommandation

In [100]:
saisie = input("Entrez un nom ou un identifiant produit : ").strip()
recommander(saisie)

,id,cluster,products,score
363,364,14,"trails vest - simplicity action - vest protects core , moves , breathes , weighs nearly . lightweight , wind - resistant 15 - denier 100 % nylon treated deluge dwr durable water repellent finish shed light rain , exceptionally breathable stretch - woven panel 91 % - recycled polyester/9 % spandex vents excess heat . high - rise tricot - lined neck provides tight seal armholes hem elastic binding . reflective heat - transfer logos reflective center - zippered pocket visible .",0.996180
95,96,14,"trails jkt - bridger ridge run normal trail - runners barkley marathons insane , find stride . jacket lightweight , wind - resistant 15 - denier 100 % nylon s treated deluge dwr durable water repellent finish shed light rain . exceptionally breathable stretch - woven underarm panels 91 % - recycled polyester/9 % spandex vent excess heat . high - rise tricot - lined neck , revised wrists improved elastic - style cuffs easily pushed drawcord hem seal chilly weather . jacket stashes internal chest pocket . reflective heat - transfer logos visible .",0.584840
280,281,14,"trails jkt - trails jacket women idea entertainment lung - busting mountain passes , reality tv shows . lightweight , wind - resistant 15 - denier 100 % nylon , s treated deluge dwr durable water repellent finish shed light rain . exceptionally breathable stretch - woven underarm panels 91 % - recycled polyester/9 % spandex vent excess heat . high - rise tricot - lined neck , revised wrists improved elastic - style cuffs easily pushed drawcord hem seal chilly weather . jacket stashes internal chest pocket . reflective heat - transfer logos visible .",0.573157
4,5,14,"alpine wind jkt - high ridges , steep ice alpine , jacket serves true best worlds staple . excels stand - shell blustery rock climbs , cool - weather trail runs high - output ski tours . , conditions ice alpine climbing , functions lightly insulated windshirt approach , frictionless midlayer s time bundle tie - . polyester ripstop shell deluge dwr durable water repellent treatment sheds snow blocks wind , smooth , lightly brushed hanging mesh liner wicks moisture , dries fast , doesn t bind baselayers . superlight stretch - woven underarm panels enhance breathability allow unimpaired arm motion , hand pockets close zippers . drawcord hem , elastic cuffs , heat - transfer reflective logo regular - coil , center - zipper dwr finish round features . updated season improved fit . recyclable common threads recycling program .",0.241709
307,308,14,"alpine wind jkt - high ridges , steep ice alpine , jacket serves true best worlds staple . excels stand - shell blustery rock climbs , cool - weather trail runs high - output ski tours . , conditions ice alpine climbing , functions lightly insulated windshirt approach , frictionless midlayer s time bundle tie - . polyester ripstop shell deluge dwr durable water repellent treatment sheds snow blocks wind , smooth , lightly brushed hanging mesh liner wicks moisture , dries fast , doesn t bind baselayers . superlight stretch - woven underarm panels enhance breathability allow unimpaired arm motion , hand pockets close zippers . drawcord hem , elastic cuffs , heat - transfer reflective logo regular - coil , center - zipper dwr finish round features . updated season improved fit . recyclable common threads recycling program .",0.241294
